# Assignment 3: Classification Models and Evaluation Metrics

## Task 1: Train and compare two classification models

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
from IPython.display import display # Import display explicitly to avoid errors

# Load and preprocess data (same steps as Assignment 2)
df = pd.read_csv('financial_customer_investment_risk_dataset (1).xls')

# Preprocessing
X = df.drop(['Customer_ID', 'High_Investment_Risk'], axis=1)
y = df['High_Investment_Risk']

# Handle missing values
num_cols = ['Age', 'Annual_Income', 'Portfolio_Value']
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

cat_cols = ['Employment_Status', 'Risk_Tolerance', 'Advisor_Contacted']
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# Encoding
le = LabelEncoder()
cat_features = ['Employment_Status', 'Risk_Tolerance', 'Advisor_Contacted', 'Previous_Investment_Loss']
for col in cat_features:
    X[col] = le.fit_transform(X[col])
y = le.fit_transform(y)

# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 1. Logistic Regression Model

In [2]:
lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

print("Logistic Regression Evaluation:")
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_pred))
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print(f"Precision: {precision_score(y_test, lr_pred):.4f}")
print(f"Recall: {recall_score(y_test, lr_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, lr_pred):.4f}")

Logistic Regression Evaluation:
Confusion Matrix:
 [[72  1]
 [ 5  0]]
Accuracy: 0.9231
Precision: 0.0000
Recall: 0.0000
F1-Score: 0.0000


### 2. SVM Model

In [3]:
svm_model = SVC(kernel='linear') # Using a linear kernel for comparison
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)

print("SVM Evaluation:")
print("Confusion Matrix:\n", confusion_matrix(y_test, svm_pred))
print(f"Accuracy: {accuracy_score(y_test, svm_pred):.4f}")
print(f"Precision: {precision_score(y_test, svm_pred):.4f}")
print(f"Recall: {recall_score(y_test, svm_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, svm_pred):.4f}")

SVM Evaluation:
Confusion Matrix:
 [[71  2]
 [ 5  0]]
Accuracy: 0.9103
Precision: 0.0000
Recall: 0.0000
F1-Score: 0.0000


## Task 2: Compare the results and interpret them

### Interpretation

1. **Which model performed better?**
   - Logistic Regression performed slightly better in terms of **Accuracy** (~92.3%) compared to SVM (~91.0%). However, both models failed to identify any 'High Risk' customers (Recall = 0), which is a common challenge with imbalanced datasets.

2. **Which metric is most important for your business problem?**
   - In financial risk, **Recall** is the most critical metric. The cost of missing a high-risk customer (False Negative) is usually much higher than the cost of misidentifying a low-risk customer (False Positive).

3. **What do false positives and false negatives mean in your dataset?**
   - **False Positive (FP)**: Predicting a customer is high-risk when they are actually low-risk. Results in unnecessary caution and potential loss of revenue from an active investor.
   - **False Negative (FN)**: Predicting a customer is low-risk when they are actually high-risk. Results in potential financial default and high risk exposure for the institution.

4. **What is one possible limitation or bias in your model?**
   - **Limitation**: The extreme **class imbalance** in the dataset causes the model to bias its predictions towards the majority 'Low Risk' class to maximize accuracy, leading to poor recall.

5. **Why should human judgment still be used?**
   - Human judgment is needed to interpret context that models cannot, such as sudden changes in a client's life or macroeconomic shifts that aren't yet reflected in historical training data. Advisors can also catch "black swan" events that models are inherently blind to.